# 04: Expanded Dataset Pipeline (138 Funds) + History Filter + Range-Based Projections

Clean, linear rebuild of the 140→138 fund expansion work, now including:
- The short-history bias fix (`starts_before_2018` filter)
- Range-based (worst / average / balanced) projected returns instead of a
  single optimistic point estimate

Loads from saved checkpoints rather than recomputing the CAGR pipeline or
retraining the model — that logic lives in `03_eda_feature_engineering.ipynb`
if it ever needs to be reproduced from scratch. This notebook is the clean
"final state" version for actually using the results.

## Known limitations (see notes at the bottom for full detail)
- `predicted_min_cagr` is **category-level, not fund-level** — every fund in
  the same category at the same `window_years` gets an identical ML
  prediction. Use `min` (historical) for fund-specific risk differentiation.
- One data-quality fix baked into the checkpoint: **Invesco India Short
  Duration Fund** (Scheme_Code 120560) had a corrupted NAV value on
  2013-04-22 that inflated its scale ~100x from that date forward. Its
  pre-2013-04-22 history was dropped and CAGR/risk figures recomputed.
- **Short-history bias**: funds whose entire NAV history starts after 2018
  only reflect an unusually strong recent bull period, producing inflated
  `mean_cagr`/`min_cagr`. Fixed below by excluding those funds from the
  recommendation pool (see the filtering section).

## Setup

In [29]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')

from calculators import compound_interest, recommend_fund, advise_investment
import importlib
import calculators
importlib.reload(calculators)
from calculators import compound_interest, recommend_fund, advise_investment

## Load checkpoints

In [30]:
results_df = pd.read_csv('../Data/external/results_df_138funds_with_predictions.csv')
final_selection = pd.read_csv('../Data/external/final_selection_140funds.csv')
nav_selected = pd.read_csv('../Data/external/nav_selected_140funds.csv', parse_dates=['date'])

print(results_df.shape)
print(results_df['Scheme_Code'].nunique(), "unique funds")
results_df.head()

(935, 7)
126 unique funds


,Scheme_Code,window_years,mean_cagr,min_cagr,max_cagr,overall_cagr,predicted_min_cagr
0,118269,1,0.165303,-0.201841,0.864233,0.140952,-0.097038
1,118269,2,0.158986,-0.052699,0.469191,0.140952,-0.010862
2,118269,3,0.154762,0.005373,0.296994,0.140952,0.029497
3,118269,4,0.155859,0.052588,0.308123,0.140952,0.056677
4,118269,5,0.157720,0.026134,0.270603,0.140952,0.063750


## Build `risk_return_df`

Rename historical columns and attach a readable fund name from `final_selection`.

In [31]:
name_lookup = final_selection[['Scheme_Code', 'Scheme_NAV_Name']].drop_duplicates()

risk_return_df = results_df.rename(columns={
    'mean_cagr': 'mean',
    'min_cagr': 'min'
})
risk_return_df = risk_return_df.merge(name_lookup, on='Scheme_Code', how='left')
risk_return_df = risk_return_df.rename(columns={'Scheme_NAV_Name': 'fund'})

print(risk_return_df.shape)
print(risk_return_df['fund'].isna().sum())
risk_return_df[['fund', 'window_years', 'mean', 'min', 'predicted_min_cagr']].head()

(935, 8)
0


,fund,window_years,mean,min,predicted_min_cagr
0,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,1,0.165303,-0.201841,-0.097038
1,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,2,0.158986,-0.052699,-0.010862
2,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,3,0.154762,0.005373,0.029497
3,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,4,0.155859,0.052588,0.056677
4,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,5,0.157720,0.026134,0.063750


## Short-history filter

**Why this exists:** funds whose entire NAV history starts after 2018 have
had all their rolling-window CAGR values computed *only* from the 2019-2024
period -- an unusually strong stretch for Indian equities (COVID recovery +
broader bull run). This inflated `mean_cagr` and `min_cagr` for these funds
in a way that doesn't reflect a full market cycle. 46% of the 138 funds
(63/138) fell into this category.

**The fix:** exclude funds whose NAV history doesn't reach back before
2018 -- i.e. require at least some exposure to a pre-2018 (weaker/mixed)
period before trusting their multi-year figures. This is a data-availability
gate, not a statistical patch -- it removes funds that structurally cannot
have a trustworthy multi-year track record, rather than adjusting numbers
for funds that remain in the pool.

In [32]:
history_span = nav_selected.groupby('Scheme_Code')['date'].agg(['min', 'max']).reset_index()
history_span['starts_before_2018'] = history_span['min'] < pd.Timestamp('2018-01-01')

risk_return_df = risk_return_df.merge(
    history_span[['Scheme_Code', 'starts_before_2018']], on='Scheme_Code', how='left'
)

risk_return_df_filtered = risk_return_df[risk_return_df['starts_before_2018'] == True].copy()

print(f"Before filter: {risk_return_df['Scheme_Code'].nunique()} funds")
print(f"After filter: {risk_return_df_filtered['Scheme_Code'].nunique()} funds")
print(risk_return_df_filtered['min'].isna().sum())  # should be 0

Before filter: 126 funds
After filter: 75 funds
0


Save the filtered dataset as its own checkpoint -- this is the dataset the tool should actually run on going forward.

In [33]:
risk_return_df_filtered.to_csv('../Data/external/risk_return_df_138funds_filtered_min7yr.csv', index=False)

## Range-based projections (worst / average / balanced)

**Why this changed:** showing only `mean_cagr`-based projected value gave a
single, optimistic-looking number (e.g. "Rs100000 becomes Rs400000 in 5
years"), which -- even for a real fund with real history -- reads like a
promise rather than a historical average. `advise_investment()` in
`calculators.py` was updated to show three figures instead:

- **Worst case** -- projected value using the risk column (`min` or
  `predicted_min_cagr`, whichever was passed in)
- **Average case** -- projected value using `mean` (the original number)
- **Balanced estimate** -- the average of the worst-case and average-case
  *final rupee amounts* (not the CAGRs -- averaging compounded results,
  not rates, since CAGR compounding is exponential)

This is now baked into `calculators.py` -- the function signature is
unchanged, so nothing else needs to be touched to use it.

In [34]:
advise_investment(risk_return_df_filtered, principal=100000, years=5, penalty=1.0, risk_column='min', n=3)

Top 3 fund(s) recommened for 5 years at penalty weight 1.0 (risk measure: min):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2455 
      Worst Case Cagr return (min)=0.0234
      Score= 0.2455

  #2: Edelweiss Mid Cap Fund - Direct Plan - Growth Option
      Expected Cagr=0.2381 
      Worst Case Cagr return (min)=0.1192
      Score= 0.2381

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2158 
      Worst Case Cagr return (min)=0.0497
      Score= 0.2158


Projected value of Rs100000 over 5 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs206013.33
      Best case (mean):       approx Rs299745.50
      Worst case (min):   approx Rs112281.16
  Edelweiss Mid Cap Fund - Direct Plan - Growth Option:
      Balanced estimate:      approx Rs233246.36
      Best case (mean):       approx Rs290909.19
      Worst case (min):   app

,fund,mean,min,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.245520,0.023438,0.245520
1,Edelweiss Mid Cap Fund - Direct Plan - Growth ...,0.238088,0.119172,0.238088
2,Axis Small Cap Fund - Direct Plan - Growth,0.215831,0.049742,0.215831


## More example calls, on the filtered dataset

**Conservative — short horizon, high penalty**

In [35]:
advise_investment(risk_return_df_filtered, principal=100000, years=1, penalty=2.0, risk_column='min', n=1)

Top 1 fund(s) recommened for 1 years at penalty weight 2.0 (risk measure: min):

  #1: ICICI Prudential Short Term Fund - Direct Plan - Growth Option
      Expected Cagr=0.0854 
      Worst Case Cagr return (min)=0.0301
      Score= 0.0854


Projected value of Rs100000 over 1 years, per recommended fund:
  ICICI Prudential Short Term Fund - Direct Plan - Growth Option:
      Balanced estimate:      approx Rs105772.14
      Best case (mean):       approx Rs108537.28
      Worst case (min):   approx Rs103007.01


,fund,mean,min,score
0,ICICI Prudential Short Term Fund - Direct Plan...,0.085373,0.03007,0.085373


**Long horizon — historical vs ML risk measure**

In [36]:
advise_investment(risk_return_df_filtered, principal=100000, years=10, penalty=1.0, risk_column='min', n=3)

Top 3 fund(s) recommened for 10 years at penalty weight 1.0 (risk measure: min):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2498 
      Worst Case Cagr return (min)=0.1969
      Score= 0.2498

  #2: Kotak Midcap Fund - Direct Plan - Growth
      Expected Cagr=0.2126 
      Worst Case Cagr return (min)=0.1668
      Score= 0.2126

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2111 
      Worst Case Cagr return (min)=0.1775
      Score= 0.2111


Projected value of Rs100000 over 10 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs766692.70
      Best case (mean):       approx Rs929877.29
      Worst case (min):   approx Rs603508.11
  Kotak Midcap Fund - Direct Plan - Growth:
      Balanced estimate:      approx Rs577449.50
      Best case (mean):       approx Rs687126.30
      Worst case (min):   approx Rs467772.70
  Axis

,fund,mean,min,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.249806,0.196929,0.249806
1,Kotak Midcap Fund - Direct Plan - Growth,0.212561,0.166819,0.212561
2,Axis Small Cap Fund - Direct Plan - Growth,0.211063,0.177538,0.211063


In [37]:
advise_investment(risk_return_df_filtered, principal=100000, years=10, penalty=1.0, risk_column='predicted_min_cagr', n=3)

Top 3 fund(s) recommened for 10 years at penalty weight 1.0 (risk measure: predicted_min_cagr):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2498 
      Worst Case Cagr return (predicted_min_cagr)=0.1459
      Score= 0.2498

  #2: Kotak Midcap Fund - Direct Plan - Growth
      Expected Cagr=0.2126 
      Worst Case Cagr return (predicted_min_cagr)=0.1087
      Score= 0.2126

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2111 
      Worst Case Cagr return (predicted_min_cagr)=0.1459
      Score= 0.2111


Projected value of Rs100000 over 10 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs660109.90
      Best case (mean):       approx Rs929877.29
      Worst case (predicted_min_cagr):   approx Rs390342.52
  Kotak Midcap Fund - Direct Plan - Growth:
      Balanced estimate:      approx Rs483869.45
      Best case (mean):  

,fund,mean,predicted_min_cagr,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.249806,0.145894,0.249806
1,Kotak Midcap Fund - Direct Plan - Growth,0.212561,0.108692,0.212561
2,Axis Small Cap Fund - Direct Plan - Growth,0.211063,0.145894,0.211063


## Notes for README / next steps

- **`predicted_min_cagr` granularity**: trained on `window_years` + one-hot
  `category` only (no fund-level feature), to avoid per-fund sparsity at
  138-fund scale. Estimates a *typical category's* downside risk, not a
  fund-specific one. Historical `min` remains the fund-specific measure.
  Possible v2: add each fund's own historical `mean_cagr` as a single
  numeric feature to differentiate within a category without reintroducing
  sparsity.
- **Data-quality fix applied**: Invesco India Short Duration Fund
  (Scheme_Code 120560), NAV data error on 2013-04-22 (~100x scale jump).
  Pre-2013-04-22 history dropped; CAGR/risk recomputed on corrected segment.
- **Short-history bias, fixed**: funds starting after 2018 excluded from
  `risk_return_df_filtered` via the `starts_before_2018` gate above. Reduced
  the usable pool from 138 to a smaller set of funds with genuine multi-cycle
  history. The unfiltered `risk_return_df` is still available if needed, but
  `risk_return_df_filtered` should be the default for real recommendations.
- **Projections now show a range** (worst / average / balanced) instead of
  a single point estimate, addressing the risk of the tool reading as an
  overconfident promise rather than a historical summary.
- **Two funds dropped entirely** during data-quality checks (Scheme_Code
  148265, 148313) — NAV history was `0.0` for every row.
- **Pending cleanup** (carried over): Phase 4/5 (original 6-fund modeling)
  still lives inside `03_eda_feature_engineering.ipynb` rather than its own
  notebook.

In [38]:
print((risk_return_df_filtered['min'] < 0).sum(), "out of", len(risk_return_df_filtered))

197 out of 744
